In [41]:
import duckdb

duckdb.sql("""
           CREATE OR REPLACE TABLE filtered AS
           SELECT message_id, channel_id, user_id, date, text_content, reply_to, n_forwards,
           views, reactions, forward_from, is_vaccine_related, language
           FROM read_json_auto('telegram.jsonl')
           WHERE text_content NOT NULL AND text_content != ''
           AND (language = 'Portuguese' OR language = 'English' OR language = 'Spanish')
           """)

# transform timestamp utc from mili to sec
duckdb.sql("UPDATE filtered SET date = date/1000")

In [42]:
duckdb.sql("""
           SELECT language, COUNT(*) FROM filtered
           GROUP BY language
           """).show()

┌────────────┬──────────────┐
│  language  │ count_star() │
│  varchar   │    int64     │
├────────────┼──────────────┤
│ Portuguese │      2331403 │
│ English    │       320762 │
│ Spanish    │        71014 │
└────────────┴──────────────┘



# Channel table

In [43]:
# the reply_to field only contains references to messages within the same channel.
# because of that, only the forward_from field was considered. 
duckdb.sql("""
           CREATE OR REPLACE VIEW channels AS
           SELECT channel_id, forward_from
           FROM filtered
           WHERE forward_from NOT NULL
           AND forward_from NOT LIKE '<USER%'
           AND channel_id != forward_from
           """)

duckdb.sql("SELECT COUNT(*) FROM channels")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       337657 │
└──────────────┘

In [44]:
# Limit the forward_from references to just the 119 channels that had their messages
# tracked on the original dataset.
# Besides, group in pairs the message channel and the original channel (channel_id and
# forward_from) to compose the future graph edges.

duckdb.sql("""
           CREATE OR REPLACE VIEW channels2 AS
           SELECT channel_id, forward_from, count(*) AS weight
           FROM channels
           WHERE forward_from IN (
                SELECT DISTINCT channel_id
                FROM channels)
           GROUP BY channel_id, forward_from
           """)
duckdb.sql("SELECT COUNT(*) FROM channels2")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         1731 │
└──────────────┘

In [45]:
duckdb.sql("COPY channels2 TO 'channels.csv' (HEADER, DELIMITER ',')")

# Tabela de conversas